# 🔒 Phishing Detection System - URL-Based Training on Google Colab

This notebook trains **two models** to detect 4 types of URLs:
- **benign** — safe URLs
- **defacement** — compromised or altered sites
- **phishing** — fraudulent links designed to steal information
- **malware** — URLs hosting malicious software

**Models:**
1. **Random Forest** (no epochs) - Traditional ML
2. **Neural Network** (10 epochs) - Deep Learning

**⚠️ Memory Optimized**: Uses data sampling to fit in Google Colab's free tier RAM

## 📋 Instructions
1. Upload your dataset files to Google Colab or mount Google Drive
2. Run all cells in order
3. Compare both models and download the best one

## 1️⃣ Install Required Libraries

In [ ]:
# Install required packages
!pip install -q scikit-learn pandas numpy matplotlib seaborn tensorflow

## 2️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras for Neural Network
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("✅ All libraries imported successfully!")
print(f"   TensorFlow version: {tf.__version__}")

## 3️⃣ Configuration - Adjust Sample Size Here

**⚠️ IMPORTANT**: To avoid RAM crashes, we'll use a sample of the data.

- **100,000 URLs**: Safe for Colab free tier (~12GB RAM)
- **200,000 URLs**: May work but risky
- **Full dataset (651K)**: Will crash on free tier

**Recommendation**: Start with 100K, increase if you have Colab Pro

In [ ]:
# Configuration
SAMPLE_SIZE = 100000  # Adjust this based on your Colab tier
USE_FULL_DATASET = False  # Set to True only if you have Colab Pro

print(f"⚙️  Configuration:")
if USE_FULL_DATASET:
    print("   Using FULL dataset (may crash on free tier!)")
else:
    print(f"   Using SAMPLE of {SAMPLE_SIZE:,} URLs (memory-safe)")
    print(f"   This is {SAMPLE_SIZE/651191*100:.1f}% of the full dataset")

## 4️⃣ Upload Dataset Files

**Option A: Upload files directly**
- Run the cell below and upload `train.csv`, `validation.csv`, and `test.csv`

**Option B: Mount Google Drive** (recommended for large files)
- Upload your files to Google Drive first
- Uncomment and run the Google Drive mount code

In [ ]:
# Option A: Upload files directly
from google.colab import files

print("📤 Please upload train.csv, validation.csv, and test.csv")
uploaded = files.upload()

# Set file paths
TRAIN_PATH = 'train.csv'
VAL_PATH = 'validation.csv'
TEST_PATH = 'test.csv'

In [ ]:
# Option B: Mount Google Drive (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')

# # Update these paths to match your Google Drive folder
# TRAIN_PATH = '/content/drive/MyDrive/PhishingDetection/data/processed/train.csv'
# VAL_PATH = '/content/drive/MyDrive/PhishingDetection/data/processed/validation.csv'
# TEST_PATH = '/content/drive/MyDrive/PhishingDetection/data/processed/test.csv'

## 5️⃣ Load and Sample Data (Memory Optimized)

In [ ]:
print("=" * 60)
print("📂 LOADING DATASETS")
print("=" * 60)

# Load datasets
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"\n📊 Original Dataset Sizes:")
print(f"   Train: {len(train_df):,} URLs")
print(f"   Validation: {len(val_df):,} URLs")
print(f"   Test: {len(test_df):,} URLs")
print(f"   Total: {len(train_df) + len(val_df) + len(test_df):,} URLs")

# Sample data if needed
if not USE_FULL_DATASET:
    print(f"\n⚠️  Sampling {SAMPLE_SIZE:,} URLs to avoid RAM crash...")
    
    # Calculate proportional sample sizes
    total_size = len(train_df) + len(val_df) + len(test_df)
    train_sample_size = int(SAMPLE_SIZE * (len(train_df) / total_size))
    val_sample_size = int(SAMPLE_SIZE * (len(val_df) / total_size))
    test_sample_size = SAMPLE_SIZE - train_sample_size - val_sample_size
    
    # Stratified sampling to maintain class distribution
    train_df = train_df.groupby('type', group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), int(train_sample_size * len(x) / len(train_df))), random_state=42)
    ).reset_index(drop=True)
    
    val_df = val_df.groupby('type', group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), int(val_sample_size * len(x) / len(val_df))), random_state=42)
    ).reset_index(drop=True)
    
    test_df = test_df.groupby('type', group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), int(test_sample_size * len(x) / len(test_df))), random_state=42)
    ).reset_index(drop=True)
    
    print(f"\n✅ Sampled Dataset Sizes:")
    print(f"   Train: {len(train_df):,} URLs")
    print(f"   Validation: {len(val_df):,} URLs")
    print(f"   Test: {len(test_df):,} URLs")
    print(f"   Total: {len(train_df) + len(val_df) + len(test_df):,} URLs")

# Show class distribution
print(f"\n📈 Train Set Distribution:")
print("-" * 60)
for url_type, count in train_df['type'].value_counts().items():
    percentage = (count / len(train_df)) * 100
    print(f"   {url_type:15s}: {count:7,} ({percentage:5.2f}%)")

# Display sample URLs
print(f"\n🔍 Sample URLs from each type:")
print("-" * 60)
for url_type in train_df['type'].unique():
    sample_url = train_df[train_df['type'] == url_type]['url'].iloc[0]
    print(f"   {url_type:15s}: {sample_url[:80]}...")

## 6️⃣ Prepare Data (URL Text Only)

In [ ]:
print("=" * 60)
print("📝 PREPARING URL TEXT DATA")
print("=" * 60)

# Extract URLs and labels
X_train_urls = train_df['url'].values
X_val_urls = val_df['url'].values
X_test_urls = test_df['url'].values

print(f"\n✅ Extracted URL text from datasets")
print(f"   Train URLs: {len(X_train_urls):,}")
print(f"   Validation URLs: {len(X_val_urls):,}")
print(f"   Test URLs: {len(X_test_urls):,}")

## 7️⃣ Vectorize URLs with TF-IDF

**TF-IDF (Term Frequency-Inverse Document Frequency)** converts URL text into numerical features by:
- Breaking URLs into character n-grams (e.g., "http", "ttp:", "tp://")
- Counting how often each n-gram appears
- Weighting by how unique each n-gram is across all URLs

In [ ]:
print("=" * 60)
print("🔤 VECTORIZING URLs WITH TF-IDF")
print("=" * 60)

print("\n⚙️  TF-IDF Configuration:")
print("   - analyzer: 'char' (character-level)")
print("   - ngram_range: (2, 5) (2 to 5 character sequences)")
print("   - max_features: 3000 (reduced from 5000 to save RAM)")
print("   - min_df: 2 (pattern must appear in at least 2 URLs)")

# Initialize TF-IDF vectorizer (reduced features to save RAM)
vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(2, 5),
    max_features=3000,  # Reduced from 5000
    min_df=2,
    lowercase=True
)

# Fit on training data and transform all sets
print("\n🚀 Fitting vectorizer on training data...")
X_train = vectorizer.fit_transform(X_train_urls)
print(f"✅ Training data vectorized: {X_train.shape}")

print("\n🔄 Transforming validation and test data...")
X_val = vectorizer.transform(X_val_urls)
X_test = vectorizer.transform(X_test_urls)

print(f"✅ Validation data vectorized: {X_val.shape}")
print(f"✅ Test data vectorized: {X_test.shape}")

print(f"\n📊 Feature Matrix Shape: {X_train.shape[0]:,} URLs × {X_train.shape[1]:,} features")

## 8️⃣ Encode Labels

In [ ]:
print("\n🏷️  Encoding labels...")

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['type'])
y_val = label_encoder.transform(val_df['type'])
y_test = label_encoder.transform(test_df['type'])

print(f"\n✅ Label classes: {list(label_encoder.classes_)}")
print(f"   Encoded as: {list(range(len(label_encoder.classes_)))}")
print(f"   Number of classes: {len(label_encoder.classes_)}")

---
# 🌲 MODEL 1: Random Forest (No Epochs)
---

## 9️⃣ Train Random Forest Model

In [ ]:
print("\n" + "=" * 60)
print("🌲 TRAINING RANDOM FOREST MODEL")
print("=" * 60)

print("\n⚙️  Model Configuration:")
print("   - n_estimators: 100")
print("   - max_depth: 20")
print("   - min_samples_split: 10")
print("   - random_state: 42")
print("   - n_jobs: -1 (use all CPU cores)")
print("   - Training: ONE-PASS (no epochs)")

# Initialize model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Train model
print("\n🚀 Training started...")
start_time = datetime.now()

rf_model.fit(X_train, y_train)

end_time = datetime.now()
rf_training_time = (end_time - start_time).total_seconds()

print(f"\n✅ Training completed in {rf_training_time:.2f} seconds ({rf_training_time/60:.2f} minutes)")

# Validation metrics
rf_val_predictions = rf_model.predict(X_val)
rf_val_accuracy = accuracy_score(y_val, rf_val_predictions)
rf_val_f1 = f1_score(y_val, rf_val_predictions, average='weighted')

print(f"\n📊 Random Forest Validation Metrics:")
print(f"   Accuracy: {rf_val_accuracy*100:.2f}%")
print(f"   F1-Score: {rf_val_f1:.4f}")

## 🔟 Evaluate Random Forest on Test Set

In [ ]:
print("\n" + "=" * 60)
print("📈 RANDOM FOREST - TEST SET EVALUATION")
print("=" * 60)

# Make predictions
rf_y_pred = rf_model.predict(X_test)

# Calculate metrics
rf_test_accuracy = accuracy_score(y_test, rf_y_pred)
rf_test_f1 = f1_score(y_test, rf_y_pred, average='weighted')

print(f"\n🎯 Random Forest Test Metrics:")
print(f"   Accuracy: {rf_test_accuracy*100:.2f}%")
print(f"   F1-Score: {rf_test_f1:.4f}")

# Classification report
print("\n📋 Classification Report:")
print("=" * 60)
print(classification_report(y_test, rf_y_pred, target_names=label_encoder.classes_))

# Confusion Matrix
rf_cm = confusion_matrix(y_test, rf_y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title(f'Random Forest Confusion Matrix\nAccuracy: {rf_test_accuracy*100:.2f}%', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

---
# 🧠 MODEL 2: Neural Network (10 Epochs)
---

## 1️⃣1️⃣ Build Neural Network Model

In [ ]:
print("\n" + "=" * 60)
print("🧠 BUILDING NEURAL NETWORK MODEL")
print("=" * 60)

# Convert sparse matrices to dense for neural network
print("\n🔄 Converting sparse matrices to dense arrays...")
X_train_dense = X_train.toarray()
X_val_dense = X_val.toarray()
X_test_dense = X_test.toarray()
print("✅ Conversion complete")

# Build neural network (smaller architecture to save RAM)
print("\n🏗️  Building Neural Network Architecture:")
print(f"   - Input Layer: {X_train_dense.shape[1]} features")
print("   - Hidden Layer 1: 128 neurons + ReLU + Dropout(0.3)")
print("   - Hidden Layer 2: 64 neurons + ReLU + Dropout(0.2)")
print("   - Output Layer: 4 neurons + Softmax")
print("   - Optimizer: Adam")
print("   - Loss: Sparse Categorical Crossentropy")
print("   - Epochs: 10")

nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_dense.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(len(label_encoder.classes_), activation='softmax')
])

nn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n📊 Model Summary:")
nn_model.summary()

## 1️⃣2️⃣ Train Neural Network (10 Epochs)

In [ ]:
print("\n" + "=" * 60)
print("🚀 TRAINING NEURAL NETWORK - 10 EPOCHS")
print("=" * 60)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Train the model
print("\n🎯 Training started with 10 epochs...\n")
start_time = datetime.now()

history = nn_model.fit(
    X_train_dense, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(X_val_dense, y_val),
    callbacks=[early_stopping],
    verbose=1
)

end_time = datetime.now()
nn_training_time = (end_time - start_time).total_seconds()

print(f"\n✅ Training completed in {nn_training_time:.2f} seconds ({nn_training_time/60:.2f} minutes)")

## 1️⃣3️⃣ Plot Training History

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='s')
ax1.set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss plot
ax2.plot(history.history['loss'], label='Train Loss', marker='o')
ax2.plot(history.history['val_loss'], label='Validation Loss', marker='s')
ax2.set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Training history plotted!")

## 1️⃣4️⃣ Evaluate Neural Network on Test Set

In [ ]:
print("\n" + "=" * 60)
print("📈 NEURAL NETWORK - TEST SET EVALUATION")
print("=" * 60)

# Make predictions
nn_y_pred_proba = nn_model.predict(X_test_dense)
nn_y_pred = np.argmax(nn_y_pred_proba, axis=1)

# Calculate metrics
nn_test_accuracy = accuracy_score(y_test, nn_y_pred)
nn_test_f1 = f1_score(y_test, nn_y_pred, average='weighted')

print(f"\n🎯 Neural Network Test Metrics:")
print(f"   Accuracy: {nn_test_accuracy*100:.2f}%")
print(f"   F1-Score: {nn_test_f1:.4f}")

# Classification report
print("\n📋 Classification Report:")
print("=" * 60)
print(classification_report(y_test, nn_y_pred, target_names=label_encoder.classes_))

# Confusion Matrix
nn_cm = confusion_matrix(y_test, nn_y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(nn_cm, annot=True, fmt='d', cmap='Greens', 
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title(f'Neural Network Confusion Matrix\nAccuracy: {nn_test_accuracy*100:.2f}%', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

---
# 📊 MODEL COMPARISON
---

## 1️⃣5️⃣ Compare Both Models

In [ ]:
print("\n" + "=" * 60)
print("⚖️  MODEL COMPARISON")
print("=" * 60)

comparison_data = {
    'Model': ['Random Forest', 'Neural Network'],
    'Test Accuracy': [f"{rf_test_accuracy*100:.2f}%", f"{nn_test_accuracy*100:.2f}%"],
    'Test F1-Score': [f"{rf_test_f1:.4f}", f"{nn_test_f1:.4f}"],
    'Training Time': [f"{rf_training_time:.2f}s", f"{nn_training_time:.2f}s"],
    'Epochs': ['N/A (one-pass)', '10'],
    'Model Type': ['Traditional ML', 'Deep Learning']
}

comparison_df = pd.DataFrame(comparison_data)
print("\n")
print(comparison_df.to_string(index=False))

# Determine winner
print("\n" + "=" * 60)
if rf_test_accuracy > nn_test_accuracy:
    print("🏆 WINNER: Random Forest")
    print(f"   Better accuracy by {(rf_test_accuracy - nn_test_accuracy)*100:.2f}%")
elif nn_test_accuracy > rf_test_accuracy:
    print("🏆 WINNER: Neural Network")
    print(f"   Better accuracy by {(nn_test_accuracy - rf_test_accuracy)*100:.2f}%")
else:
    print("🤝 TIE: Both models have equal accuracy")
print("=" * 60)

## 1️⃣6️⃣ Save Both Models

In [ ]:
print("\n" + "=" * 60)
print("💾 SAVING BOTH MODELS")
print("=" * 60)

# Save Random Forest
with open('random_forest_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print("\n✅ Random Forest saved: random_forest_model.pkl")

# Save Neural Network
nn_model.save('neural_network_model.h5')
print("✅ Neural Network saved: neural_network_model.h5")

# Save vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
print("✅ TF-IDF vectorizer saved: tfidf_vectorizer.pkl")

# Save label encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("✅ Label encoder saved: label_encoder.pkl")

print("\n📥 Download all 4 files to use the models!")

## 1️⃣7️⃣ Download Model Files

In [ ]:
from google.colab import files

print("📥 Downloading model files...")
files.download('random_forest_model.pkl')
files.download('neural_network_model.h5')
files.download('tfidf_vectorizer.pkl')
files.download('label_encoder.pkl')
print("\n✅ All files downloaded!")

## ✅ Summary

### Memory Optimization:
- **Sampled dataset**: 100K URLs (15% of full dataset)
- **Reduced features**: 3000 instead of 5000
- **Smaller network**: 2 hidden layers instead of 3
- **Result**: Fits in Colab free tier (~12GB RAM)

### Training Approach:
- **No manual feature extraction** ✅
- **Direct URL text training** using TF-IDF vectorization
- **Character n-grams** (2-5 characters) capture URL patterns

### Models Trained:
1. **Random Forest** - No epochs (one-pass training)
2. **Neural Network** - 10 epochs of iterative learning

### To Use Full Dataset:
1. Get **Google Colab Pro** (more RAM)
2. Set `USE_FULL_DATASET = True` in cell 3
3. Increase `max_features` back to 5000

### Next Steps:
1. Compare accuracy on sampled vs full dataset
2. Try different sample sizes (150K, 200K)
3. Use the best model for deployment